<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [15]</a>'.</span>

In [1]:
from vivarium import Artifact, InteractiveContext
import pandas as pd, numpy as np, os

In [2]:
! pip list | grep vivarium

# [software verion + hash . date]

In [3]:
! pip freeze | grep vivarium

In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from vivarium import InteractiveContext, Artifact

In [5]:
import vivarium_gates_mncnh
from vivarium.framework.configuration import build_model_specification
from pathlib import Path

from vivarium_gates_mncnh.constants.data_values import (
    SIMULATION_EVENT_NAMES,
    COLUMNS,
    PREGNANCY_OUTCOMES,
    PIPELINES,
)

path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
custom_model_specification = build_model_specification(path)
custom_model_specification.configuration.input_data.input_draw_number = 60

custom_model_specification.configuration.population.population_size = 20_000 * 3

included_components = custom_model_specification.components.vivarium_gates_mncnh.components
included_components

['Hemoglobin()',
 'AnemiaInterventionPropensity()',
 'AgelessPopulation("population.scaling_factor")',
 'Pregnancy()',
 'ResultsStratifier()',
 'ANCAttendance()',
 'Ultrasound()',
 'MaternalDisorder("maternal_obstructed_labor_and_uterine_rupture")',
 'MaternalDisorder("maternal_hemorrhage")',
 'MaternalDisorder("maternal_sepsis_and_other_maternal_infections")',
 'ResidualMaternalDisorders()',
 'AbortionMiscarriageEctopicPregnancy()',
 'MaternalDisordersBurden()',
 'LBWSGRisk()',
 "LBWSGRiskEffect('cause.all_causes.all_cause_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_with_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_without_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.cause_specific_mortality_risk')",
 "PretermBirth('neonatal_preterm_birth_with_r

In [6]:
artifact_path = custom_model_specification.configuration.input_data.artifact_path
art = Artifact(artifact_path)

artifact_path

'/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model37.0/ethiopia.hdf'

In [7]:
draw_num = custom_model_specification.configuration.input_data.input_draw_number
draw = 'draw_' + str(draw_num)
draw

'draw_60'

CHECK: puerperal sepsis hemoglobin effect size in the artifact does not vary by sex, age, or year.

Suite: artifact tests.

Type: precise assert.

In [8]:
sepsis_on_hemoglobin_shift = art.load("cause.maternal_sepsis_and_other_maternal_infections.hemoglobin_shift")[draw]
assert len(sepsis_on_hemoglobin_shift) == 2  # early and late postpartum
sepsis_on_hemoglobin_shift

postpartum_period
early_postpartum   -13.247920
late_postpartum     -3.042343
Name: draw_60, dtype: float64

In [9]:
def check_hemoglobin_shift(sim: InteractiveContext):
    pop = sim.get_population([COLUMNS.MATERNAL_SEPSIS, COLUMNS.PREGNANCY_OUTCOME])
    # We model maternal sepsis on live and still births only
    live_births = pop.loc[
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.LIVE_BIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.STILLBIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.FULL_TERM_OUTCOME)
    ]
    has_sepsis = live_births[COLUMNS.MATERNAL_SEPSIS]
    sepsis_idx = live_births.index[has_sepsis]
    no_sepsis_idx = live_births.index[~has_sepsis]
    if len(sepsis_idx) == 0:
        return
    # Get hemoglobin exposure values from the pipeline
    hgb = sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE).loc[live_births.index]
    mean_hgb_sepsis = hgb.loc[sepsis_idx].mean()
    mean_hgb_no_sepsis = hgb.loc[no_sepsis_idx].mean()

    return {'step': sim._clock.step_name, 'mean_hgb_sepsis': mean_hgb_sepsis, 'mean_hgb_no_sepsis': mean_hgb_no_sepsis}

In [10]:
def run_all_steps(sim: InteractiveContext):
    step_shifts = []
    while sim.get_number_of_steps_remaining() > 0:
        step_shift = check_hemoglobin_shift(sim)
        if step_shift:
            step_shifts.append(step_shift)
        sim.step()
    return pd.DataFrame(step_shifts)

In [11]:
def run_to_step(sim: InteractiveContext, step_name: str):
    while sim._clock.step_name != step_name:
        sim.step()
    return sim

In [12]:
sim = InteractiveContext(custom_model_specification)
step_shifts = run_all_steps(sim)

2026-07-24 10:09:17.598 | INFO     | simulation_1-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model37.0/ethiopia.hdf.


2026-07-24 10:09:17.599 | INFO     | simulation_1-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].


2026-07-24 10:09:17.599 | INFO     | simulation_1-artifact_manager:82 - Artifact additional filter terms are None.


2026-07-24 10:09:41.139 | WARNING  | simulation_1-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']


2026-07-24 10:09:41.140 | WARNING  | simulation_1-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']


...

2026-07-24 10:10:36.528 | INFO     | simulation_1 - vivarium.framework.engine:280 - 2025-01-18 00:00:00


2026-07-24 10:10:39.554 | INFO     | simulation_1 - vivarium.framework.engine:280 - 2025-01-19 00:00:00


2026-07-24 10:10:56.222 | INFO     | simulation_1 - vivarium.framework.engine:280 - 2025-01-20 00:00:00


2026-07-24 10:11:02.297 | INFO     | simulation_1 - vivarium.framework.engine:280 - 2025-01-21 00:00:00


2026-07-24 10:11:02.851 | WARNING  | simulation_1-population_manager:747 - The 'postpartum_depression.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'postpartum_depression.incidence_risk.paf'.


In [13]:
step_shifts["observed_diff"] = step_shifts["mean_hgb_sepsis"] - step_shifts["mean_hgb_no_sepsis"]
step_shifts

,step,mean_hgb_sepsis,mean_hgb_no_sepsis,observed_diff
0,residual_maternal_disorders,106.976969,120.502504,-13.525535
1,abortion_miscarriage_ectopic_pregnancy,106.976969,120.502504,-13.525535
2,mortality,106.976969,120.502504,-13.525535
3,early_neonatal_mortality,93.729049,120.502504,-26.773456
4,late_neonatal_mortality,103.934626,120.502504,-16.567878
5,postpartum_depression,106.976969,120.502504,-13.525535


In [14]:
step_shifts = step_shifts.set_index("step")[["observed_diff"]].diff().iloc[1:]
step_shifts

,observed_diff
step,
abortion_miscarriage_ectopic_pregnancy,0.000000
mortality,0.000000
early_neonatal_mortality,-13.247920
late_neonatal_mortality,10.205578
postpartum_depression,3.042343


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [15]:
assert np.isclose(step_shifts.loc["mortality"], 0)
assert np.isclose(step_shifts.loc["early_postpartum"], sepsis_on_hemoglobin_shift["early_postpartum"])
assert np.isclose(step_shifts.loc["late_postpartum"], sepsis_on_hemoglobin_shift["late_postpartum"] - sepsis_on_hemoglobin_shift["early_postpartum"])
assert np.isclose(step_shifts.loc["early_neonatal_mortality"], -sepsis_on_hemoglobin_shift["late_postpartum"])
assert np.isclose(step_shifts.loc["late_neonatal_mortality"], 0)

KeyError: 'early_postpartum'

In [ ]:
sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE)  # we should be able to get this data at the end of the sim